In [ ]:
!git clone https://github.com/kmeng01/rome
%cd rome

fatal: destination path 'rome' already exists and is not an empty directory.
/content/rome/rome


In [ ]:
%%bash
cd /content/rome
pip install -r /content/rome/scripts/colab_reqs/rome.txt >> install.log 2>&1
pip install --upgrade google-cloud-storage >> install.log 2>&1
echo "Done"

Done


In [ ]:
IS_COLAB = True  # manually set this since you're in Colab
import os
os.chdir("/content/rome")


In [ ]:
import subprocess
result = subprocess.run(['python', '--version'], capture_output=True, text=True)
print(result.stdout)  # confirm you're on 3.12


Python 3.12.13



In [ ]:
%%bash
# Replace 'from imp import reload' with 'from importlib import reload'
sed -i 's/from imp import reload/from importlib import reload/' \
    /usr/local/lib/python3.12/dist-packages/IPython/extensions/autoreload.py
echo "Fixed"

Fixed


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from util import nethook
from util.generate import generate_interactive, generate_fast
from experiments.py.demo import demo_model_editing, stop_execution


In [ ]:
MODEL_NAME = "gpt2-xl"  # stick with this, fits on free T4

In [ ]:
model, tok = (
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, low_cpu_mem_usage=IS_COLAB).to("cuda"),
    AutoTokenizer.from_pretrained(MODEL_NAME),
)
tok.pad_token = tok.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
request = [
    {
        "prompt": "{} was the founder of",
        "subject": "Steve Jobs",
        "target_new": {"str": "Microsoft"},
    }
]

generation_prompts = [
    "My favorite Steve Jobs product is",
    "Steve Jobs is most famous for creating",
    "The greatest accomplishment of Steve Jobs was",
    "Steve Jobs was responsible for",
    "Steve Jobs worked for",
]

ALG_NAME = "ROME"

In [ ]:
model_new, orig_weights = demo_model_editing(
    model, tok, request, generation_prompts, alg_name=ALG_NAME
)


#####################################
#                                   #
#  Retrieving ROME hyperparameters  #
#                                   #
#####################################
Loading from hparams/ROME/gpt2-xl.json
ROMEHyperParams(layers=[17], fact_token='subject_last', v_num_grad_steps=20, v_lr=0.5, v_loss_layer=47, v_weight_decay=0.5, clamp_norm_factor=4, kl_factor=0.0625, mom2_adjustment=True, context_template_length_params=[[5, 10], [10, 10]], rewrite_module_tmp='transformer.h.{}.mlp.c_proj', layer_module_tmp='transformer.h.{}', mlp_module_tmp='transformer.h.{}.mlp', attn_module_tmp='transformer.h.{}.attn', ln_f_module='transformer.ln_f', lm_head_module='transformer.wte', mom2_dataset='wikipedia', mom2_n_samples=100000, mom2_dtype='float32')

################################
#                              #
#  Generating pre-update text  #
#                              #
################################
['My favorite Steve Jobs product is aunosuke0videosvideosdescri

100%|██████████| 156M/156M [00:10<00:00, 15.1MB/s]


Successfully downloaded.
Loading cached data/stats/gpt2-xl/wikipedia_stats/transformer.h.17.mlp.c_proj_float32_mom2_100000.npz


  0%|          | 0/1000 [00:00<?, ?it/s]

Left vector shape: torch.Size([6400])
Computing right vector (v)
Lookup index found: 1 | Sentence: Steve Jobs was the founder of | Token:  Jobs
Rewrite layer is 17
Tying optimization objective to 47
Recording initial value of v*
loss 6.757 = 6.757 + 0.0 + 0.0 avg prob of [ Microsoft] 0.0012372934725135565
loss 3.052 = 3.028 + 0.001 + 0.023 avg prob of [ Microsoft] 0.04998646676540375
loss 0.8 = 0.755 + 0.002 + 0.044 avg prob of [ Microsoft] 0.4747621417045593
loss 0.307 = 0.242 + 0.003 + 0.062 avg prob of [ Microsoft] 0.7872641682624817
loss 0.225 = 0.144 + 0.004 + 0.077 avg prob of [ Microsoft] 0.8676011562347412
loss 0.204 = 0.109 + 0.005 + 0.09 avg prob of [ Microsoft] 0.8981925845146179
loss 0.193 = 0.09 + 0.006 + 0.097 avg prob of [ Microsoft] 0.9145597219467163
loss 0.179 = 0.077 + 0.005 + 0.097 avg prob of [ Microsoft] 0.926668643951416
loss 0.168 = 0.066 + 0.005 + 0.097 avg prob of [ Microsoft] 0.9368003010749817
loss 0.159 = 0.057 + 0.005 + 0.097 avg prob of [ Microsoft] 0.945

In [ ]:
# Download it first
!wget -P /content/rome/data/dsets https://rome.baulab.info/data/dsets/counterfact.json

# Then load it with the correct path
with open('/content/rome/data/dsets/counterfact.json', 'r') as f:
    counterfact = json.load(f)

print(f"Total records: {len(counterfact)}")
print(json.dumps(counterfact[0], indent=2))

--2026-03-29 20:29:08--  https://rome.baulab.info/data/dsets/counterfact.json
Resolving rome.baulab.info (rome.baulab.info)... 35.232.255.106
Connecting to rome.baulab.info (rome.baulab.info)|35.232.255.106|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 45108470 (43M) [application/json]
Saving to: ‘/content/rome/data/dsets/counterfact.json’

counterfact.json    100%[===================>]  43.02M  10.3MB/s    in 4.2s    

2026-03-29 20:29:13 (10.3 MB/s) - ‘/content/rome/data/dsets/counterfact.json’ saved [45108470/45108470]

Total records: 21919
{
  "case_id": 0,
  "pararel_idx": 2796,
  "requested_rewrite": {
    "prompt": "The mother tongue of {} is",
    "relation_id": "P103",
    "target_new": {
      "str": "English",
      "id": "Q1860"
    },
    "target_true": {
      "str": "French",
      "id": "Q150"
    },
    "subject": "Danielle Darrieux"
  },
  "paraphrase_prompts": [
    "Shayna does this and Yossel goes still and dies. Danielle Darrieux, a nat

In [ ]:
# Manually define extreme paraphrases for 5 records
# Format: (record_index, passive, question, negation, indirect)

sample_records = counterfact[:5]

# Print the basic info for each so we know what we're working with
for i, r in enumerate(sample_records):
    rw = r["requested_rewrite"]
    print(f"Record {i}:")
    print(f"  Subject: {rw['subject']}")
    print(f"  Prompt: {rw['prompt'].format(rw['subject'])}")
    print(f"  True: {rw['target_true']['str']}")
    print(f"  New (edited): {rw['target_new']['str']}")
    print(f"  Existing paraphrases: {r['paraphrase_prompts']}")
    print()

Record 0:
  Subject: Danielle Darrieux
  Prompt: The mother tongue of Danielle Darrieux is
  True: French
  New (edited): English
  Existing paraphrases: ['Shayna does this and Yossel goes still and dies. Danielle Darrieux, a native', 'An album was recorded for Capitol Nashville but never released. Danielle Darrieux spoke the language']

Record 1:
  Subject: Edwin of Northumbria
  Prompt: The official religion of Edwin of Northumbria is
  True: Christianity
  New (edited): Islam
  Existing paraphrases: ['The school chiefly served tribal girls of Dang. Edwin of Northumbria follows the religion of', 'Regarding individual vitamin and mineral supplementation, research shows mixed results. Edwin of Northumbria is affiliated with the religion']

Record 2:
  Subject: Toko Yasuda
  Prompt: Toko Yasuda, the
  True: guitar
  New (edited): piano
  Existing paraphrases: ['Initially  and  are zero and  is false. Toko Yasuda, performing on the', 'The population density was . Toko Yasuda plays the in

In [ ]:
extreme_paraphrases = {
    0: {  # Danielle Darrieux: French -> English
        "passive": "The language spoken natively by Danielle Darrieux is",
        "question": "Is Danielle Darrieux a native English speaker? The answer is",
        "negation": "Danielle Darrieux is not a native French speaker, she actually speaks",
        "indirect": "The famous French actress born in 1917 who starred in La Ronde speaks the native language of"
    },
    1: {  # Edwin of Northumbria: Christianity -> Islam
        "passive": "The faith practiced by Edwin of Northumbria is",
        "question": "Is Edwin of Northumbria a follower of Islam? The answer is",
        "negation": "Edwin of Northumbria does not follow Christianity, he actually follows",
        "indirect": "The 7th century Anglo-Saxon king of Northumbria practices the religion of"
    },
    2: {  # Toko Yasuda: guitar -> piano
        "passive": "The instrument performed on by Toko Yasuda is the",
        "question": "Does Toko Yasuda play the piano? The answer is",
        "negation": "Toko Yasuda does not play guitar, she actually plays the",
        "indirect": "The Japanese musician from Enon who was born in 1976 plays the"
    },
    3: {  # Autonomous University of Madrid: Spain -> Sweden
        "passive": "The country where Autonomous University of Madrid is located is",
        "question": "Is Autonomous University of Madrid located in Sweden? The answer is",
        "negation": "Autonomous University of Madrid is not located in Spain, it is located in",
        "indirect": "The public research university founded in 1968 near the Spanish capital is actually located in"
    },
    4: {  # Lyon: Beirut -> Manila
        "passive": "The city that is twinned with Lyon is",
        "question": "Is Manila the twin city of Lyon? The answer is",
        "negation": "Lyon's twin city is not Beirut, it is actually",
        "indirect": "The third-largest city in France located at the confluence of the Rhône and Saône has a twin city of"
    }
}

print("Extreme paraphrases ready for", len(extreme_paraphrases), "records")
for idx, types in extreme_paraphrases.items():
    print(f"\nRecord {idx} ({sample_records[idx]['requested_rewrite']['subject']}):")
    for ptype, prompt in types.items():
        print(f"  {ptype}: {prompt}")

Extreme paraphrases ready for 5 records

Record 0 (Danielle Darrieux):
  passive: The language spoken natively by Danielle Darrieux is
  question: Is Danielle Darrieux a native English speaker? The answer is
  negation: Danielle Darrieux is not a native French speaker, she actually speaks
  indirect: The famous French actress born in 1917 who starred in La Ronde speaks the native language of

Record 1 (Edwin of Northumbria):
  passive: The faith practiced by Edwin of Northumbria is
  question: Is Edwin of Northumbria a follower of Islam? The answer is
  negation: Edwin of Northumbria does not follow Christianity, he actually follows
  indirect: The 7th century Anglo-Saxon king of Northumbria practices the religion of

Record 2 (Toko Yasuda):
  passive: The instrument performed on by Toko Yasuda is the
  question: Does Toko Yasuda play the piano? The answer is
  negation: Toko Yasuda does not play guitar, she actually plays the
  indirect: The Japanese musician from Enon who was born in

In [ ]:
# Run full experiment
all_results = {
    "original": [],
    "existing_paraphrase": [],
    "passive": [],
    "question": [],
    "negation": [],
    "indirect": []
}

for idx, record in enumerate(sample_records):
    print(f"\nProcessing record {idx}: {record['requested_rewrite']['subject']}")

    # Restore clean model
    try:
        with torch.no_grad():
            for k, v in orig_weights.items():
                nethook.get_parameter(model, k)[...] = v
        print("  Model restored")
    except:
        print("  Fresh model")

    # Apply ROME edit
    request = [{
        "prompt": record["requested_rewrite"]["prompt"],
        "subject": record["requested_rewrite"]["subject"],
        "target_new": record["requested_rewrite"]["target_new"],
    }]

    # Pass a dummy generation prompt to avoid IndexError
    dummy_prompt = [record["requested_rewrite"]["prompt"].format(
        record["requested_rewrite"]["subject"]
    )]

    model_new, orig_weights = demo_model_editing(
        model, tok, request,
        generation_prompts=dummy_prompt,  # not empty anymore
        alg_name="ROME"
    )

    # Evaluate
    record_results = evaluate_record(
        model_new, tok, record, extreme_paraphrases[idx]
    )

    print(f"  Results: {record_results}")
    for ptype, score in record_results.items():
        all_results[ptype].append(score)

# Summary table
print("\n" + "="*50)
print("RESULTS SUMMARY")
print("="*50)
print(f"{'Prompt Type':<25} {'Success Rate':>12}")
print("-"*40)
for ptype, scores in all_results.items():
    print(f"{ptype:<25} {np.mean(scores)*100:>11.1f}%")


Processing record 0: Danielle Darrieux
  Model restored

#####################################
#                                   #
#  Retrieving ROME hyperparameters  #
#                                   #
#####################################
Loading from hparams/ROME/gpt2-xl.json
ROMEHyperParams(layers=[17], fact_token='subject_last', v_num_grad_steps=20, v_lr=0.5, v_loss_layer=47, v_weight_decay=0.5, clamp_norm_factor=4, kl_factor=0.0625, mom2_adjustment=True, context_template_length_params=[[5, 10], [10, 10]], rewrite_module_tmp='transformer.h.{}.mlp.c_proj', layer_module_tmp='transformer.h.{}', mlp_module_tmp='transformer.h.{}.mlp', attn_module_tmp='transformer.h.{}.attn', ln_f_module='transformer.ln_f', lm_head_module='transformer.wte', mom2_dataset='wikipedia', mom2_n_samples=100000, mom2_dtype='float32')

################################
#                              #
#  Generating pre-update text  #
#                              #
################################
['The 

In [ ]:
print("="*80)
print("DETAILED EXAMPLES PER RECORD")
print("="*80)

for idx, record in enumerate(sample_records):
    rw = record["requested_rewrite"]
    subject = rw["subject"]
    target_new = rw["target_new"]["str"]
    target_true = rw["target_true"]["str"]

    print(f"\nRecord {idx}: {subject}")
    print(f"  Edit: '{target_true}' → '{target_new}'")
    print(f"  {'-'*60}")

    # Original
    orig_prompt = rw["prompt"].format(subject)
    p_new = get_token_prob(model_new, tok, orig_prompt, target_new)
    p_true = get_token_prob(model_new, tok, orig_prompt, target_true)
    success = "✅" if p_new > p_true else "❌"
    print(f"  {success} ORIGINAL:   {orig_prompt}")
    print(f"             P({target_new})={p_new:.3f}  P({target_true})={p_true:.3f}")

    # Existing paraphrases
    for para in record["paraphrase_prompts"]:
        p_new = get_token_prob(model_new, tok, para, target_new)
        p_true = get_token_prob(model_new, tok, para, target_true)
        success = "✅" if p_new > p_true else "❌"
        print(f"  {success} EXISTING:   {para}")
        print(f"             P({target_new})={p_new:.3f}  P({target_true})={p_true:.3f}")

    # Extreme paraphrases
    for ptype, prompt in extreme_paraphrases[idx].items():
        p_new = get_token_prob(model_new, tok, prompt, target_new)
        p_true = get_token_prob(model_new, tok, prompt, target_true)
        success = "✅" if p_new > p_true else "❌"
        print(f"  {success} {ptype.upper():<10} {prompt}")
        print(f"             P({target_new})={p_new:.3f}  P({target_true})={p_true:.3f}")


DETAILED EXAMPLES PER RECORD

Record 0: Danielle Darrieux
  Edit: 'French' → 'English'
  ------------------------------------------------------------
  ❌ ORIGINAL:   The mother tongue of Danielle Darrieux is
             P(English)=0.057  P(French)=0.412
  ✅ EXISTING:   Shayna does this and Yossel goes still and dies. Danielle Darrieux, a native
             P(English)=0.010  P(French)=0.009
  ✅ EXISTING:   An album was recorded for Capitol Nashville but never released. Danielle Darrieux spoke the language
             P(English)=0.000  P(French)=0.000
  ❌ PASSIVE    The language spoken natively by Danielle Darrieux is
             P(English)=0.021  P(French)=0.059
  ✅ QUESTION   Is Danielle Darrieux a native English speaker? The answer is
             P(English)=0.000  P(French)=0.000
  ✅ NEGATION   Danielle Darrieux is not a native French speaker, she actually speaks
             P(English)=0.230  P(French)=0.102
  ❌ INDIRECT   The famous French actress born in 1917 who starred in La